In [1]:
import os
import glob
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset, WeightedRandomSampler
from transformers import AutoVideoProcessor, AutoModel
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import numpy as np

# MPS - Mac GPU Acceleration
assert torch.backends.mps.is_available(), "Pytorch MPS backend is not available. Please ensure you're running on a compatible Mac with M4 Max and have the latest PyTorch version installed."
device = torch.device("mps")
dtype = torch.float32 # MPS works best with float32 precision

print(f"🚀 M4 Max GPU Activated | Device: {device}")

🚀 M4 Max GPU Activated | Device: mps


In [2]:
from torchvision.transforms import v2
import torchvision.transforms.functional as F

class RugbyVideoDataset(Dataset):
    def __init__(self, root_dir, processor, num_frames=16, is_train=False):
        self.root_dir = root_dir
        self.processor = processor
        self.num_frames = num_frames
        self.is_train = is_train # Đánh dấu cờ Augmentation
        
        # 8 labels
        self.labels = ["Tackle", "Ruck", "Maul", "General Play", "Out of Play", "Scrum", "Lineout", "Goal Kick"]
        self.label2id = {label: i for i, label in enumerate(self.labels)}
        
        self.video_paths = []
        self.video_labels = []
        
        # Load paths
        for label in self.labels:
            folder_path = os.path.join(root_dir, label)
            all_files = os.listdir(folder_path) if os.path.exists(folder_path) else []
            for f in all_files:
                if f.lower().endswith(('.mp4', '.mov', '.avi')):
                    self.video_paths.append(os.path.join(folder_path, f))
                    self.video_labels.append(self.label2id[label])
                
        print(f"✅ Loaded {len(self.video_paths)} videos from {len(self.labels)} classes (is_train={self.is_train}).")

        # === ĐỊNH NGHĨA AUGMENTATION KHÔNG GIAN ===
        if self.is_train:
            self.spatial_transform = v2.Compose([
                v2.RandomHorizontalFlip(p=0.5),
                v2.RandomApply([v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)], p=0.8),
                v2.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3), value='random')
            ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.video_labels[idx]
        
        cap = cv2.VideoCapture(video_path)
        pil_frames = []
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(1, total_frames // self.num_frames)
        
        # === TEMPORAL AUGMENTATION ===
        start_offset = 0
        if self.is_train and total_frames > self.num_frames:
            max_offset = min(step, total_frames - (self.num_frames - 1) * step)
            start_offset = torch.randint(0, max_offset, (1,)).item() if max_offset > 0 else 0

        for i in range(self.num_frames):
            frame_idx = min(start_offset + i * step, total_frames - 1)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret: break
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_frames.append(Image.fromarray(rgb_frame))
        cap.release()
        
        while len(pil_frames) < self.num_frames:
            pil_frames.append(pil_frames[-1] if pil_frames else Image.new('RGB', (224, 224)))

        # === SPATIAL AUGMENTATION ===
        if self.is_train:
            tensor_frames = torch.stack([F.pil_to_tensor(img) for img in pil_frames])
            tensor_frames = self.spatial_transform(tensor_frames)
            frames_for_processor = list(tensor_frames)
        else:
            frames_for_processor = pil_frames

        inputs = self.processor(frames_for_processor, return_tensors="pt")
        pixel_values = inputs['pixel_values_videos'].squeeze(0)
        
        return pixel_values, torch.tensor(label, dtype=torch.long)

In [3]:
class RugbyLinearProbe(nn.Module):
    def __init__(self, model_name="facebook/vjepa2-vitl-fpc64-256", num_classes=8):
        super().__init__()
        print(f"⏳ Khởi tạo V-JEPA 2: {model_name}...")
        self.backbone = AutoModel.from_pretrained(model_name)
        
        # Freeze backbone parameters
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        embed_dim = self.backbone.config.hidden_size if hasattr(self.backbone.config, 'hidden_size') else 1024
        
        # Fine tuning head: LayerNorm + Dropout + Linear
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.5), # Chống Overfitting
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, pixel_values_videos):
        # Since backbone is frozen, we can disable gradient calculation for efficiency
        with torch.no_grad():
            outputs = self.backbone(pixel_values_videos=pixel_values_videos)
            video_features = outputs.last_hidden_state.mean(dim=1) # Average Pooling
            
        # Push feature through the trainable head
        logits = self.head(video_features)
        return logits

# Initialize processor and model
processor = AutoVideoProcessor.from_pretrained("facebook/vjepa2-vitl-fpc64-256")
model = RugbyLinearProbe(num_classes=8).to(device, dtype=dtype)
print("✅ Model is ready with frozen backbone and trainable head.")

⏳ Khởi tạo V-JEPA 2: facebook/vjepa2-vitl-fpc64-256...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

✅ Model is ready with frozen backbone and trainable head.


In [4]:
DATA_DIR = "mydata"
NUM_FRAMES = 32
BATCH_SIZE = 4
NUM_WORKERS = 0

dataset_for_train = RugbyVideoDataset(root_dir=DATA_DIR, processor=processor, num_frames=NUM_FRAMES, is_train=True)
dataset_for_val = RugbyVideoDataset(root_dir=DATA_DIR, processor=processor, num_frames=NUM_FRAMES, is_train=False)

indices = list(range(len(dataset_for_train)))
labels = dataset_for_train.video_labels

train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

train_subset = Subset(dataset_for_train, train_idx) # Có Augmentation
val_subset = Subset(dataset_for_val, val_idx)       # KHÔNG Augmentation

train_labels = [dataset_for_train.video_labels[i] for i in train_idx]
class_counts = np.bincount(train_labels, minlength=8)
class_weights = 1.0 / (class_counts + 1e-5) 
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True
)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"✅ Training: {len(train_subset)} video | Validation: {len(val_subset)} video")

✅ Loaded 366 videos from 8 classes (is_train=True).
✅ Loaded 366 videos from 8 classes (is_train=False).
✅ Training: 292 video | Validation: 74 video


In [5]:
import copy
import os

EPOCHS = 6
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.head.parameters(), lr=LEARNING_RATE)

print("🚀 Starting/Resuming Linear Probing Training...")

best_val_acc = 0.0

if os.path.exists("best_rugby_head.pth"):
    model.head.load_state_dict(torch.load("best_rugby_head.pth"))
    print("🔄 Found 'best_rugby_head.pth'")
    best_val_acc = 71.62 
else:
    print("⚠️ Can not found previous save. Starting fresh training.")

best_model_weights = copy.deepcopy(model.head.state_dict())
patience = 5
epochs_no_improve = 0
best_epoch = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for pixel_values, batch_labels in loop:
        pixel_values = pixel_values.to(device, dtype=dtype)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(pixel_values)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_labels.size(0)
        correct += predicted.eq(batch_labels).sum().item()
        
        loop.set_postfix(loss=loss.item(), acc=100.*correct/total)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for pixel_values, batch_labels in val_loader:
            pixel_values = pixel_values.to(device, dtype=dtype)
            batch_labels = batch_labels.to(device)
            outputs = model(pixel_values)
            _, predicted = outputs.max(1)
            val_total += batch_labels.size(0)
            val_correct += predicted.eq(batch_labels).sum().item()
            
    val_acc = 100. * val_correct / val_total
    print(f"📊 Results Epoch {epoch+1}: Train Loss: {total_loss/len(train_loader):.4f} | Val Accuracy: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 4
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model.head.state_dict())
        torch.save(best_model_weights, "best_rugby_head.pth")
        print(f"⭐ Best model updated with Val Acc: {best_val_acc:.2f}%\n")
    else:
        epochs_no_improve += 1
        print(f"⚠️ No improvement. Patience: {epochs_no_improve}/{patience}\n")
        
        if epochs_no_improve >= patience:
            print("🛑 Early stopping activated! Stopping training to prevent overfitting.")
            break

print(f"🎉 Completed! Best model updated at Epoch {best_epoch} with Val Acc: {best_val_acc:.2f}%\n")
print("You can load the best model with: model.head.load_state_dict(torch.load('best_rugby_head.pth'))")

🚀 Starting/Resuming Linear Probing Training...
🔄 Found 'best_rugby_head.pth'


Epoch 1/6 [Train]:   0%|          | 0/73 [00:00<?, ?it/s]/Users/tntran2/miniforge3/envs/mllm_rugby/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Epoch 1/6 [Train]: 100%|██████████| 73/73 [14:09<00:00, 11.64s/it, acc=74, loss=0.806]  


📊 Results Epoch 1: Train Loss: 0.7469 | Val Accuracy: 63.51%
⚠️ No improvement. Patience: 1/5



Epoch 2/6 [Train]: 100%|██████████| 73/73 [14:45<00:00, 12.13s/it, acc=71.6, loss=0.186]


📊 Results Epoch 2: Train Loss: 0.7932 | Val Accuracy: 74.32%
⭐ Best model updated with Val Acc: 74.32%



...

📊 Results Epoch 4: Train Loss: 0.7590 | Val Accuracy: 70.27%
⚠️ No improvement. Patience: 2/5



Epoch 5/6 [Train]: 100%|██████████| 73/73 [14:52<00:00, 12.23s/it, acc=72.9, loss=0.241]


📊 Results Epoch 5: Train Loss: 0.7605 | Val Accuracy: 70.27%
⚠️ No improvement. Patience: 3/5



Epoch 6/6 [Train]: 100%|██████████| 73/73 [14:53<00:00, 12.24s/it, acc=71.2, loss=0.977]


📊 Results Epoch 6: Train Loss: 0.7553 | Val Accuracy: 68.92%
⚠️ No improvement. Patience: 4/5

🎉 Completed! Best model updated at Epoch 5 with Val Acc: 74.32%

You can load the best model with: model.head.load_state_dict(torch.load('best_rugby_head.pth'))


# Linear Probing (Recoverable Version)

In [ ]:
import copy
import os
import torch
from tqdm import tqdm

TOTAL_EPOCHS = 15 # Tổng số Epoch tối đa bạn muốn train
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.head.parameters(), lr=LEARNING_RATE)

print("🚀 Starting ")

best_val_acc = 0.0
start_epoch = 0
patience = 5
epochs_no_improve = 0

# Tên file checkpoint chính thức
checkpoint_path = "rugby_checkpoint.pth"

# ==========================================
# 1. KIỂM TRA VÀ TỰ ĐỘNG KHÔI PHỤC (DYNAMIC RECOVERY)
# ==========================================
if os.path.exists(checkpoint_path):
    print(f"🔄 Found '{checkpoint_path}'. Restoring process...")
    checkpoint = torch.load(checkpoint_path)
    
    # Nạp lại toàn bộ trí nhớ (Không có bất kỳ con số hardcode nào)
    model.head.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    best_val_acc = checkpoint['best_val_acc']
    start_epoch = checkpoint['epoch']
    
    print(f"✅ Successfully resumed from Epoch {start_epoch}. Best Val Acc so far: {best_val_acc:.2f}%")
else:
    print("⚠️ Cant find checkpoint. Starting fresh training.")

best_epoch = start_epoch
best_model_weights = copy.deepcopy(model.head.state_dict())

# ==========================================
# 2. VÒNG LẶP TRAIN CHUYÊN NGHIỆP
# ==========================================
for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS} [Train]")
    for pixel_values, batch_labels in loop:
        pixel_values = pixel_values.to(device, dtype=dtype)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(pixel_values)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_labels.size(0)
        correct += predicted.eq(batch_labels).sum().item()
        
        loop.set_postfix(loss=loss.item(), acc=100.*correct/total)

    # --- ĐÁNH GIÁ (VALIDATION) ---
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for pixel_values, batch_labels in val_loader:
            pixel_values = pixel_values.to(device, dtype=dtype)
            batch_labels = batch_labels.to(device)
            outputs = model(pixel_values)
            _, predicted = outputs.max(1)
            val_total += batch_labels.size(0)
            val_correct += predicted.eq(batch_labels).sum().item()
            
    val_acc = 100. * val_correct / val_total
    print(f"📊 Results Epoch {epoch+1}: Train Loss: {total_loss/len(train_loader):.4f} | Val Accuracy: {val_acc:.2f}%")

    # ==========================================
    # 3. LƯU CHECKPOINT THÔNG MINH
    # ==========================================
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        epochs_no_improve = 0
        
        # Đóng gói tất cả biến động vào từ điển
        checkpoint = {
            'epoch': best_epoch,
            'best_val_acc': best_val_acc,
            'model_state_dict': model.head.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
        }
        # Ghi đè file cũ bằng bản tốt hơn
        torch.save(checkpoint, checkpoint_path)
        print(f"⭐ New checkpoint saved at epoch {best_epoch} with Val Acc: {best_val_acc:.2f}%\n")
    else:
        epochs_no_improve += 1
        print(f"⚠️ No improvement. Patience: {epochs_no_improve}/{patience}\n")
        
        if epochs_no_improve >= patience:
            print("🛑 Early stopping activated! Stopping training to prevent overfitting.")
            break

print(f"🎉 Completed! Best accuracy is: {best_val_acc:.2f}% at Epoch {best_epoch}")

🚀 Start training/recovering...
🔄 Tìm thấy file trọng số cũ 'best_rugby_head.pth'...
✅ Old weights loaded.


Epoch 4 [Train]: 100%|██████████| 73/73 [14:30<00:00, 11.93s/it, acc=73.6, loss=0.39]  


📊 Results Epoch 4: Train Loss: 0.7471 | Val Accuracy: 81.08%
⭐ New checkpoint saved at epoch 4 with Val Acc: 81.08%



Epoch 5 [Train]: 100%|██████████| 73/73 [14:44<00:00, 12.12s/it, acc=75.7, loss=0.747] 


📊 Results Epoch 5: Train Loss: 0.7323 | Val Accuracy: 77.03%
⚠️ No improvement. Patience: 1/5



Epoch 6 [Train]: 100%|██████████| 73/73 [15:15<00:00, 12.54s/it, acc=76, loss=0.146]   


📊 Results Epoch 6: Train Loss: 0.6850 | Val Accuracy: 71.62%
⚠️ No improvement. Patience: 2/5



Epoch 7 [Train]: 100%|██████████| 73/73 [15:12<00:00, 12.50s/it, acc=70.9, loss=0.74]  


📊 Results Epoch 7: Train Loss: 0.8015 | Val Accuracy: 74.32%
⚠️ No improvement. Patience: 3/5



Epoch 8 [Train]:   8%|▊         | 6/73 [01:16<14:08, 12.66s/it, acc=66.7, loss=0.394]